In [ ]:
from pathlib import Path
import os 
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils import get_s3
import pandas as pd 
import geopandas as gpd
import json 
import matplotlib.pyplot as plt
import numpy as np
import folium 
import itertools
from shapely.ops import linemerge
from shapely import union_all
from functions_utils import *

In [ ]:
PATH_DATI_TRANSITI = Path(os.getcwd()) 
assert PATH_DATI_TRANSITI.exists(), f"Path {PATH_DATI_TRANSITI} does not exist"
PATH_DESCRIZIONI = PATH_DATI_TRANSITI / "descrizioni_traffico_veicolare"
PATH_DATI_STRADE = PATH_DATI_TRANSITI / 'dati_strade' 

### CLASSE 

| "CODCLASSE" | "DESCRIZIONE" |
| -- | -- |
| 1 | "Motocicli" |
| 2 | "Autovetture e monovolumi" |
| 3 | "Autovetture e monovolumi con rimorchio" |
| 4 | "Furgoni" |
| 5 | "Autocarro medio (fino a 8.7 m)" |
| 6 | "Autocarro grande (da 8.7 m)" |
| 7 | "Autocarro con rimorchio" |
| 8 | "Trattore con semirimorchio" |
| 9 | "Autobus" |

### DIREZIONI
"CODDIREZIONE" | "DESCRIZIONE"
| -- | -- |
1 | "Chilometriche crescenti"
2 | "Chilometriche decrescenti"

### VELOCITA'
|"CODVELOCITA"|"DESCRIZIONE"|
| -- | -- |
|1 |"0-20"|
|2 |"20-30"|
|3 |"30-40"|
|4 |"40-50"|
|5 |"50-60"|
|6 |"60-70"|
|7 |"70-80"|
|8 |"80-90"|
|9 |"90-100"|
|10 |"100-110"|
|11 |"110-120"|
|12 |"120-130"|
|13 |"130-140"|
|14 |">140"|

In [ ]:
dati_trasporto_veicolare = pd.read_parquet( PATH_DESCRIZIONI / "dati_transito_veicolare.parquet")
dati_trasporto_veicolare['DATA'] = pd.to_datetime(dati_trasporto_veicolare['DATA'], format='%d/%m/%Y')

dati_trasporto_veicolare['PUNTO'] = dati_trasporto_veicolare['PUNTO'].astype('category')
dati_trasporto_veicolare['DIREZIONE'] = dati_trasporto_veicolare['DIREZIONE'].astype('category')
dati_trasporto_veicolare['CLASSE'] = dati_trasporto_veicolare['CLASSE'].astype('category')
dati_trasporto_veicolare['VELOCITA'] = dati_trasporto_veicolare['VELOCITA'].astype('category')
top_memory_objects(globals())

In [ ]:
descrizione_classi_csv = pd.read_csv(PATH_DESCRIZIONI / "descrizione_classi_veicolari.csv")
descrizione_direzioni_csv = pd.read_csv(PATH_DESCRIZIONI / "descrizione_direzioni.csv")
descrizione_velocita_csv = pd.read_csv(PATH_DESCRIZIONI / "descrizione_velocita.csv")

In [ ]:
descrizione_velocita_dict = descrizione_velocita_csv.set_index('CODVELOCITA')['DESCRIZIONE'].to_dict()
descrizione_classi_dict = descrizione_classi_csv.set_index('CODCLASSE')['DESCRIZIONE']
descrizione_direzioni_csv = descrizione_direzioni_csv.set_index('CODDIREZIONE')['DESCRIZIONE']

In [ ]:
top_memory_objects(globals())

In [ ]:
# dati_trasporto_veicolare['DATA'] = pd.to_datetime(dati_trasporto_veicolare['DATA'], format='%d/%m/%Y')
# dati_trasporto_veicolare['DATA_COMPLETA'] = dati_trasporto_veicolare['DATA'] + pd.to_timedelta(dati_trasporto_veicolare['ORA'].astype(int), unit='h')
print(dati_trasporto_veicolare.memory_usage(deep=True))
print(f"\nTotale: {dati_trasporto_veicolare.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# Vedi i tipi di dato
print(dati_trasporto_veicolare.dtypes)

In [ ]:
dati_trasporto_veicolare.PUNTO.unique()
dati_trasporto_veicolare

Il dataframe contiene i dati di passaggi, ora per ora, per ogni classe di veicolo, direzione e velocita' di passaggio.

In [ ]:
dati_trasporto_veicolare[(dati_trasporto_veicolare['DATA'] == '2022-01-01') & (dati_trasporto_veicolare['ORA'] == 0)].groupby(['PUNTO', 'CLASSE', 'VELOCITA', 'DIREZIONE']).count().DATA.unique()
dati_trasporto_veicolare.PUNTO.unique()

In [ ]:
viabilita = PATH_DATI_STRADE / 'viabilita'
gdf_viab= gpd.read_file(viabilita / 'viabilita_gestione_pat_v.shp')
gdf = gpd.GeoDataFrame(gdf_viab, geometry="geometry")

In [ ]:
lista_strade = PATH_DATI_STRADE / 'lista_strade.xls'
lista_strade_df = pd.read_excel(lista_strade, header = 1)

strade_complete = PATH_DATI_STRADE / 'strade_subset.xls'
strade_complete_df = pd.read_excel(strade_complete, header = 1)

punti_traffico = PATH_DATI_STRADE / 'punti_traffico.xls'
punti_traffico_df = pd.read_excel(punti_traffico, header = 1)

gdf_red = gdf[gdf['str_cd'].isin(strade_complete_df['id lrs'].unique())]

# gdf_red['geometry'].apply(lambda x : x.geom_type).unique()
gdf_red

In [ ]:
ss42 = gdf_red[gdf_red["str_cd"] == 20004200]
print(len(ss42))
campiglio = gdf[gdf['cod_ser'] == 'SS 239']
gdf_red.cod_ser.unique()

In [ ]:
campiglio
merged_campiglio = linemerge(campiglio['geometry'].union_all())
lines = sorted(merged_campiglio.geoms, key=lambda l: l.coords[0][0])

## Appendix (and more inspections)

In [ ]:
lista_strade_df['id LRS'].unique()
lista_strade_df = lista_strade_df.rename(columns = {'id LRS':'id_lrs'})
lista_strade_df

In [ ]:
strade_complete_df['id lrs'].unique()
strade_complete_df = strade_complete_df.rename(columns = {'id lrs':'id_lrs'})
strade_complete_df

In [ ]:
punti_traffico_df = punti_traffico_df.rename(columns = {'LOCALITA\'': 'LOCALITA'})
punti_traffico_df

In [ ]:
# Nota: .explore() di Geopandas converte automaticamente i CRS in EPSG:4326 per Folium,
# ma allineare i CRS prima è sempre una buona pratica.
mappa_finale = create_map_start_end(gdf_red, strade_complete_df)  # geodf_apt
mappa_finale

In [ ]:
# --- 1. SPOSTIAMO LE INFO DI INIZIO STRADA IN UN DIZIONARIO ---
km_inizio_strade = dict(zip(strade_complete_df['codice servizio'], strade_complete_df['progressiva di inizio strada']))

gdf_strade_unite = gdf_red.dissolve(by='cod_ser')  # raggruppare per 'cod_ser' (es. SS 42)
gdf_strade_unite['geometry'] = gdf_strade_unite['geometry'].apply(linemerge)  # linemerge unisce i segmenti adiacenti in un'unica LineString continua

punti_traffico_df['geometry'] = punti_traffico_df.apply(
    lambda row: compute_loc_spira(row, gdf_strade_unite, km_inizio_strade), 
    axis=1
)
punti_traffico_df = gpd.GeoDataFrame(punti_traffico_df, geometry='geometry')
punti_traffico_df.set_crs(gdf_red.crs, inplace=True)

In [ ]:
gdf_strade_web = gdf_red.to_crs(epsg=4326)
gdf_spire_web = punti_traffico_df.to_crs(epsg=4326)

m = gdf_strade_web.explore(
    column='nome',             
    cmap='Set1',             
    style_kwds=dict(weight=4),
    tooltip=['nome', 'cod_ser'], # Cosa vedi quando passi il mouse sopra la strada
    popup=True,              
    name="Rete Stradale"       
)

# Aggiungiamo le SPIRE sulla mappa
gdf_spire_web.explore(
    m=m,                     
    color='darkblue',               # Colore dei pallini
    marker_kwds=dict(
        radius=7, 
        fill=True,
        color='black',         # Bordo del pallino
        weight=1
    ),
    tooltip='LOCALITA',       
    popup=['STRADA', 'KM'],
    name="Spire di Rilevamento"
)

folium.LayerControl().add_to(m)
m.save("mappa_interattiva_spire.html")

In [ ]:
punti_traffico_df = punti_traffico_df.rename( columns={'ID_strada': 'ID_strada'})
mg = strade_complete_df .merge(punti_traffico_df, on = 'codice servizio' )
mg = mg[ ['codice servizio', 'denominazione', 'progressiva di inizio strada', 'LOCALITA', 'progressiva di fine strada', 'localita di inizio strada', 'localita di fine strada', 'KM', 'geometry']]
mg['distanza'] = mg['KM'] - mg['progressiva di inizio strada'] 
mg 